# Verification of claims (appendix)

Each section restates a claim from the text, computes the check from the collected data (`analysis/data` for the `01`-`08` sweep, `kubedeadline-experiments/analysis/data` for the baselines) and saves a LaTeX table to `analysis/results/appendix_claims/`. Claims that need logs that were not collected are listed at the end with their status.

In [1]:
import pandas as pd, numpy as np, os, json, glob, re
from pathlib import Path

OUT = Path("os-experiments/analysis/results/appendix_claims"); OUT.mkdir(parents=True, exist_ok=True)
PERIOD_US = {"tight": 10000, "soft": 100000}
LABELS = {"01": "isolcpus removed", "02": "nohz_full removed", "03": "rcu_nocbs removed",
          "04": "systemd containment removed", "05": "IRQ steering removed",
          "06": "boot-params bundle removed", "08": "frequency pinning removed"}
SCENARIO_ORDER = ["01", "02", "03", "04", "05", "06", "08"]
BASELINE_LABEL = "baseline (full isolation)"
SMT_LABEL = "SMT enabled"
SCALE_LBL = {"soft": "soft (100 ms)", "tight": "tight (10 ms)"}

frames = []
for s in SCENARIO_ORDER:
    for w in ["model1", "model5"]:
        d = pd.read_csv(f"os-experiments/analysis/data/{s}-{w}.csv")
        d["scenario"], d["workload"] = s, w
        frames.append(d)
all_df = pd.concat(frames, ignore_index=True)
all_df["label"] = all_df["scenario"].map(LABELS)

bl = []
for w, d_ in {"model1": "model1", "model5": "model5-phys", "model5-smt": "model5-smt"}.items():
    d = pd.read_csv(f"kubedeadline-experiments/analysis/data/{d_}.csv")
    d["workload"] = w; d["label"] = {"model1": BASELINE_LABEL, "model5": BASELINE_LABEL, "model5-smt": SMT_LABEL}[w]
    bl.append(d)
baseline_df = pd.concat(bl, ignore_index=True)
# one frame with every configuration; baselines get their own label
full = pd.concat([all_df, baseline_df], ignore_index=True)
for c in ("R_wall_us", "C_cputime_us"):
    full[c.replace("_wall_us","_ms").replace("_cputime_us","_ms")] = full[c] / 1000
full["R_ms"] = full["R_wall_us"] / 1000; full["C_ms"] = full["C_cputime_us"] / 1000
ORDER = [BASELINE_LABEL] + [LABELS[s] for s in SCENARIO_ORDER]

def save_tex(df, name, caption, colfmt=None, float_format="%.2f", index=True):
    tex = df.to_latex(index=index, na_rep="--", escape=True, float_format=float_format, caption=caption,
                      label=f"tab:{name}", multicolumn=True, multicolumn_format="c", column_format=colfmt)
    tex = re.sub(r"\\cline\{(\d+)-(\d+)\}", r"\\cmidrule{\1-\2}", tex)
    tex = tex.replace("\\begin{tabular}", "\\resizebox{\\textwidth}{!}{%\n\\begin{tabular}", 1).replace("\\end{tabular}", "\\end{tabular}%\n}", 1)
    (OUT / f"{name}.tex").write_text(tex); print("saved", name)
    return df
print(len(full), "rows;", sorted(full.label.unique()))

7300000 rows; ['IRQ steering removed', 'SMT enabled', 'baseline (full isolation)', 'boot-params bundle removed', 'frequency pinning removed', 'isolcpus removed', 'nohz_full removed', 'rcu_nocbs removed', 'systemd containment removed']


## Claim 1: the slower single-core group is a difference in execution speed (host), not a setting effect

Median **execution time C** per configuration (single-core). If the CPU itself was slower, C differs between configurations at the same U, not only R. `slow` = baseline, isolcpus, systemd, IRQ steering removed; `fast` = the other four.

In [2]:
SLOW = {BASELINE_LABEL, LABELS["01"], LABELS["04"], LABELS["05"]}
m1 = full[(full.workload == "model1") & full.U.isin([0.1, 0.5, 0.7]) & full.label.notna()]
c1 = m1.groupby(["label", "scale", "U"])["C_ms"].median().unstack(["scale", "U"])
c1 = c1.reindex([l for l in ORDER if l in c1.index])
c1.columns = [f"{SCALE_LBL[s]}, U={u:g}" for s, u in c1.columns]
c1.insert(0, "group", ["slow" if l in SLOW else "fast" for l in c1.index])
ratio = (c1[c1.group == "slow"].drop(columns="group").mean() / c1[c1.group == "fast"].drop(columns="group").mean())
ratio.name = "slow / fast (mean of group medians)"
c1_out = pd.concat([c1, pd.DataFrame([["--"] + list(ratio.values)], index=[ratio.name], columns=c1.columns)])
c1_num = c1.drop(columns="group")
c1_num.loc[ratio.name] = ratio.values
c1_num.insert(0, "group", list(c1.group) + ["--"])
save_tex(c1_num, "claim1_median_C", "Claim 1: median execution time C (ms), single-core. The last row is the ratio between the slow group and the fast group.")
c1_num

saved claim1_median_C


,group,"soft (100 ms), U=0.1","soft (100 ms), U=0.5","soft (100 ms), U=0.7","tight (10 ms), U=0.1","tight (10 ms), U=0.5","tight (10 ms), U=0.7"
label,,,,,,,
baseline (full isolation),slow,8.229816,41.054100,55.701242,0.791600,3.941412,5.531785
isolcpus removed,slow,8.328602,41.698276,55.096458,0.775271,3.865659,5.380223
nohz_full removed,fast,8.407251,34.619241,48.066845,0.699971,3.486500,4.882833
rcu_nocbs removed,fast,6.284633,35.223817,47.753558,0.695971,3.489552,4.867462
systemd containment removed,slow,8.271385,41.083248,54.516005,0.789774,3.942746,5.502489
IRQ steering removed,slow,8.329205,41.330734,55.776550,0.787523,3.922408,5.536481
boot-params bundle removed,fast,6.269345,34.747947,49.372861,0.688119,3.472277,4.812984
frequency pinning removed,fast,6.220699,34.330996,49.721687,0.673871,3.373805,4.729477
slow / fast (mean of group medians),--,1.219892,1.188914,1.134291,1.140045,1.133850,1.137783


## Claim 2: misses are concentrated in round 1

Share of all misses that fall in round 1, for the three configurations named in the text (single-core), and the raw miss counts per round at U = 0.1. Whether a reboot preceded round 1 is not in the data (see the last section).

In [3]:
sel = [LABELS["01"], LABELS["02"], LABELS["04"]]
d2 = all_df[(all_df.workload == "model1") & all_df.label.isin(sel) & all_df.U.isin([0.1, 0.5, 0.7, 0.94])]
per = d2.groupby(["label", "scale", "U", "round"])["deadline_miss"].sum().unstack("round").fillna(0).astype(int)
per["total"] = per.sum(axis=1)
per["round 1 share (%)"] = np.where(per["total"] > 0, 100 * per[1] / per["total"], np.nan)
share = per["round 1 share (%)"].unstack("U").reindex(sel, level=0)
share.columns = [f"U={u:g}" for u in share.columns]
share.index = [f"{a}, {SCALE_LBL[b]}" for a, b in share.index]
save_tex(share, "claim2_round1_share", "Claim 2: share (\\%) of the misses that fall in round 1, single-core; -- = no misses.", float_format="%.1f")
u01 = per.xs(0.1, level="U")[[1, 2, 3, 4, "total"]]
u01.columns = [f"round {c}" if c != "total" else "total" for c in u01.columns]
u01.index = [f"{a}, {SCALE_LBL[b]}" for a, b in u01.index]
save_tex(u01, "claim2_misses_per_round_U01", "Claim 2: deadline misses per round at $U=0.1$, single-core.", float_format="%.0f")
display(share); u01

saved claim2_round1_share
saved claim2_misses_per_round_U01


,U=0.1,U=0.5,U=0.7,U=0.94
"isolcpus removed, soft (100 ms)",82.862903,98.611111,100.000000,NaN
"isolcpus removed, tight (10 ms)",97.540984,94.366197,90.000000,NaN
"nohz_full removed, soft (100 ms)",56.481481,NaN,NaN,NaN
"nohz_full removed, tight (10 ms)",0.000000,0.000000,33.333333,0.0
"systemd containment removed, soft (100 ms)",0.000000,33.333333,0.000000,NaN
"systemd containment removed, tight (10 ms)",53.658537,0.000000,14.285714,0.0


,round 1,round 2,round 3,round 4,total
"isolcpus removed, soft (100 ms)",411,81,0,4,496
"isolcpus removed, tight (10 ms)",238,2,1,3,244
"nohz_full removed, soft (100 ms)",61,6,39,2,108
"nohz_full removed, tight (10 ms)",0,19,5,1,25
"systemd containment removed, soft (100 ms)",0,5,22,83,110
"systemd containment removed, tight (10 ms)",44,5,1,32,82


## Claim 3: missed jobs were slowed down, not delayed

For missed jobs: the excess of their C over the configuration's median C, and the median of R - C (time not spent executing: preemption, softirqs, queueing) for missed jobs versus all jobs. Large R - C would mean the jobs were delayed rather than slowed.

In [4]:
rows = []
for (lab, scale, u), g in all_df[(all_df.workload == "model1") & all_df.label.isin(sel) & all_df.U.isin([0.1, 0.5, 0.7])].groupby(["label", "scale", "U"]):
    miss = g[g.deadline_miss == 1]
    if miss.empty: continue
    medC = g.C_cputime_us.median() / 1000
    rows.append({"configuration": lab, "scale": SCALE_LBL[scale], "U": f"{u:g}", "misses": len(miss),
                 "median C (ms)": medC, "missed-job C excess (%)": 100 * (miss.C_cputime_us.median() / 1000 / medC - 1),
                 "median R-C all (ms)": (g.R_wall_us - g.C_cputime_us).median() / 1000,
                 "median R-C missed (ms)": (miss.R_wall_us - miss.C_cputime_us).median() / 1000})
c3 = pd.DataFrame(rows).set_index(["configuration", "scale", "U"])
save_tex(c3, "claim3_slowed_vs_delayed", "Claim 3: execution-time excess and non-executing time (R-C) of missed jobs, single-core.")
c3

saved claim3_slowed_vs_delayed


misses  median C (ms)  \
configuration               scale         U                            
isolcpus removed            soft (100 ms) 0.1     496       8.328602   
                                          0.5     288      41.698276   
                                          0.7      88      55.096458   
                            tight (10 ms) 0.1     244       0.775271   
                                          0.5     142       3.865658   
                                          0.7      80       5.380223   
nohz_full removed           soft (100 ms) 0.1     108       8.407251   
                            tight (10 ms) 0.1      25       0.699971   
                                          0.5       5       3.486500   
                                          0.7       3       4.882833   
systemd containment removed soft (100 ms) 0.1     110       8.271386   
                                          0.5       3      41.083248   
                                          0.7       1      54.516005   
                            tight (10 ms) 0.1      82       0.789774   
                                          0.5      26       3.942746   
                                          0.7       7       5.502489   

                                               missed-job C excess (%)  \
configuration               scale         U                              
isolcpus removed            soft (100 ms) 0.1                25.141374   
                                          0.5                25.496743   
                                          0.7                30.175587   
                            tight (10 ms) 0.1                24.113890   
                                          0.5                36.788920   
                                          0.7                36.280318   
nohz_full removed           soft (100 ms) 0.1                27.664881   
                            tight (10 ms) 0.1                64.032367   
                                          0.5                50.611502   
                                          0.7                55.943297   
systemd containment removed soft (100 ms) 0.1                33.452116   
                                          0.5                26.321049   
                                          0.7                40.326431   
                            tight (10 ms) 0.1                25.073059   
                                          0.5                35.998388   
                                          0.7                27.426334   

                                               median R-C all (ms)  \
configuration               scale         U                          
isolcpus removed            soft (100 ms) 0.1             0.118323   
                                          0.5             0.111236   
                                          0.7             0.060306   
                            tight (10 ms) 0.1             0.033557   
                                          0.5             0.031833   
                                          0.7             0.031642   
nohz_full removed           soft (100 ms) 0.1             0.121159   
                            tight (10 ms) 0.1             0.025053   
                                          0.5             0.024978   
                                          0.7             0.024311   
systemd containment removed soft (100 ms) 0.1             0.240260   
                                          0.5             0.116192   
                                          0.7             0.242651   
                            tight (10 ms) 0.1             0.032290   
                                          0.5             0.030610   
                                          0.7             0.029800   

                                               median R-C missed (ms)  
configuration               scale         U                            
isolcpus rem

## Claim 5: SMT placement and slowdown

Median C under SMT versus the multi-core baseline (physical cores) at the same U, and the CPU placement recorded per cell (`placement.json`: `RT_CPUSET`, `pair_type`).

In [5]:
sm = full[(full.workload.isin(["model5", "model5-smt"])) & (full.label.isin([BASELINE_LABEL, SMT_LABEL]))]
c5 = sm.groupby(["label", "scale", "U"])["C_ms"].median().unstack("label")
c5 = c5[[BASELINE_LABEL, SMT_LABEL]]
c5["SMT / baseline"] = c5[SMT_LABEL] / c5[BASELINE_LABEL]
c5 = c5[c5.index.get_level_values("U").isin([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.94])]
c5.index = [f"{SCALE_LBL[s]}, U={u:g}" for s, u in c5.index]
c5.columns = ["median C baseline (ms)", "median C SMT (ms)", "SMT / baseline"]
save_tex(c5, "claim5_smt_C", "Claim 5: median execution time C of the multi-core baseline and of the SMT configuration.", float_format="%.2f")

pl = []
for cfg, pat in [("baseline (physical)", "kubedeadline-experiments/results/model5-phys_round*/*/U*/placement.json"),
                 ("SMT", "kubedeadline-experiments/results/model5-smt_round*/*/U*/placement.json")]:
    for f in glob.glob(pat):
        j = json.load(open(f)); parts = Path(f).parts
        pl.append({"configuration": cfg, "scale": SCALE_LBL[parts[-3]], "U": float(parts[-2][1:]),
                   "RT_CPUSET": j.get("target_RT_CPUSET"), "pair_type": j.get("pair_type")})
pl = pd.DataFrame(pl)
if not pl.empty:
    pl = (pl.groupby(["configuration", "scale", "U"]).agg(RT_CPUSET=("RT_CPUSET", lambda s: ", ".join(sorted(set(map(str, s))))),
          pair_type=("pair_type", lambda s: ", ".join(sorted(set(map(str, s)))))).reset_index())
    pl["U"] = pl["U"].map(lambda u: f"{u:g}")
    pl_soft = pl[pl.scale == SCALE_LBL["soft"]].drop(columns="scale")
    save_tex(pl_soft, "claim5_placement", "Claim 5: CPU set and pair type recorded per cell (soft scale; tight is identical unless stated in the data).", index=False)
    display(pl_soft)
else:
    print("no placement.json found")
c5

saved claim5_smt_C
saved claim5_placement


,configuration,U,RT_CPUSET,pair_type
0,SMT,0.1,2-3,sibling
1,SMT,0.2,2-3,sibling
2,SMT,0.3,2-3,sibling
3,SMT,0.4,2-3,sibling
4,SMT,0.5,2-3,sibling
5,SMT,0.6,2-3,sibling
6,SMT,0.7,2-3,sibling
7,SMT,0.8,2-3,sibling
8,SMT,0.9,2-3,sibling
9,SMT,0.94,2-3,sibling


,median C baseline (ms),median C SMT (ms),SMT / baseline
"soft (100 ms), U=0.1",6.277465,7.192060,1.145695
"soft (100 ms), U=0.2",14.632363,16.165059,1.104747
"soft (100 ms), U=0.3",20.767523,24.151865,1.162963
"soft (100 ms), U=0.4",26.974471,29.056966,1.077202
"soft (100 ms), U=0.5",35.272599,50.024057,1.418213
"soft (100 ms), U=0.6",41.608730,57.565878,1.383505
"soft (100 ms), U=0.7",50.095119,59.332347,1.184394
"soft (100 ms), U=0.8",55.885681,73.257583,1.310847
"soft (100 ms), U=0.9",62.598613,82.573061,1.319088
"soft (100 ms), U=0.94",66.855713,102.103475,1.527221


## Claim 6: the one-period tail at tight U = 0.1 in almost every multi-core configuration

p99.9 of the response time at tight U = 0.1, multi-core. Band from the text: 8.3-10.1 ms.

In [6]:
t6 = full[(full.workload.isin(["model5", "model5-smt"])) & (full.scale == "tight") & (full.U == 0.1)]
c6 = t6.groupby("label")["R_ms"].agg(median="median", p99_9=lambda x: np.percentile(x, 99.9)).reindex(ORDER + [SMT_LABEL]).dropna(how="all")
c6["in band 8.3-10.1 ms"] = np.where(c6.p99_9.between(8.3, 10.1), "yes", "no")
core = c6.loc[[l for l in ORDER if l in c6.index]]
n_in = (core["in band 8.3-10.1 ms"] == "yes").sum()
save_tex(c6, "claim6_p999_tight_U01", f"Claim 6: p99.9 and median response time (ms), multi-core, tight, $U=0.1$; {n_in} of the {len(core)} single-sweep configurations (baseline and the seven ablations, SMT excluded) fall in 8.3--10.1 ms.")
print(f"{n_in} of {len(core)} configurations in band (SMT excluded)"); c6

saved claim6_p999_tight_U01
6 of 8 configurations in band (SMT excluded)


,median,p99_9,in band 8.3-10.1 ms
label,,,
baseline (full isolation),0.755950,8.408708,yes
isolcpus removed,0.740268,8.752559,yes
nohz_full removed,0.733272,8.312453,yes
rcu_nocbs removed,0.806705,17.767647,no
systemd containment removed,0.744760,9.765732,yes
IRQ steering removed,0.748052,10.046215,yes
boot-params bundle removed,0.743610,1.233209,no
frequency pinning removed,0.753443,9.363659,yes
SMT enabled,0.839844,10.148984,no


## Supporting checks already made in the analysis (kept for the appendix)

In [7]:
# soft cells last ~10x longer than tight cells; multi-second stalls only in soft
dur = (full.groupby(["workload", "label", "scale", "U", "round"])["release_us"].agg(lambda x: (x.max() - x.min()) / 60e6)
       .groupby(["scale"]).median().rename("median duration per round (min)").to_frame())
stalls = (full[(full.R_wall_us > 1e6)].groupby(["workload", "scale"]).size().unstack("scale").fillna(0).astype(int))
stalls.columns = [f"jobs with R > 1 s, {SCALE_LBL[c]}" for c in stalls.columns]
dur.index = [SCALE_LBL[i] for i in dur.index]
save_tex(dur, "support_round_duration", "Median wall-clock duration of one round (minutes).")
save_tex(stalls, "support_multisecond_stalls", "Number of jobs with a response time above 1 s, per workload and scale.", float_format="%.0f")
display(dur); stalls

saved support_round_duration
saved support_multisecond_stalls


,median duration per round (min)
soft (100 ms),8.331667
tight (10 ms),0.833333


,"jobs with R > 1 s, soft (100 ms)","jobs with R > 1 s, tight (10 ms)"
workload,,
model1,4,1
model5,11,0


## Claims that cannot be verified from the collected data

These need logs that were not recorded (per-job CPU, kernel logs, `lscpu`, `cpufreq`, interrupt counters, reboot times).

In [8]:
na = pd.DataFrame([
 ("1 (cause: host)", "CPU model / node per configuration", "not logged per run; C differs systematically (see above), cause undetermined"),
 ("2 (reboot before round 1)", "reboot times per configuration", "not in the data; round-1 concentration is checked above"),
 ("4 (co-location)", "sched\\_getcpu() per job", "not logged; keep as hypothesis, future work"),
 ("7 (cache sizes)", "lscpu on the VM, sieve memory size", "platform fact, not in the data"),
 ("8 (frequency on Azure)", "cpufreq presence, cpupower output", "not in the data"),
 ("9 (device/network activity)", "/proc/interrupts, /proc/softirqs", "not logged"),
 ("10 (RT CPU vs housekeeping sibling)", "thread\\_siblings\\_list", "not in the data"),
], columns=["claim", "needed", "status"])
tex = na.to_latex(index=False, escape=False, caption="Claims that cannot be verified from the collected data.", label="tab:claims_unverifiable", column_format="p{4cm}p{5cm}p{5.5cm}")
(OUT / "claims_not_verifiable.tex").write_text(tex); na

,claim,needed,status
0,1 (cause: host),CPU model / node per configuration,not logged per run; C differs systematically (...
1,2 (reboot before round 1),reboot times per configuration,not in the data; round-1 concentration is chec...
2,4 (co-location),sched\_getcpu() per job,"not logged; keep as hypothesis, future work"
3,7 (cache sizes),"lscpu on the VM, sieve memory size","platform fact, not in the data"
4,8 (frequency on Azure),"cpufreq presence, cpupower output",not in the data
5,9 (device/network activity),"/proc/interrupts, /proc/softirqs",not logged
6,10 (RT CPU vs housekeeping sibling),thread\_siblings\_list,not in the data
